In [0]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys

In [0]:
sys.path.append("..")

from src.visualization.visualization import plots, plot_rul_by_engine
from src.preprocessing.data_loader import cmapss_loader_pandas

Gerar o catalogo com os dados brutos

In [0]:
# Caminhos dos arquivos
train_path = "/Volumes/workspace/default/cmapss/train_FD001.txt"
test_path = "/Volumes/workspace/default/cmapss/test_FD001.txt"
rul_path = "/Volumes/workspace/default/cmapss/RUL_FD001.txt"

# Importando os arquivos (espaço em branco como delimitador)
df_train_raw, df_test_raw, df_rul_raw = cmapss_loader_pandas(
    train_path=train_path,
    test_path=test_path,
    test_rul_path=rul_path,
)

In [0]:
display(df_train_raw.head())
display(df_test_raw.head())
display(df_rul_raw.head())

# Pré-processamento


### Criando os RULs para Treino

O artigo usa um modelo de degradação linear por partes com RUL constante inicial (Re):


In [0]:
def create_rul_labels(df, Re, clip_at_zero=True):
    """Cria os rótulos RUL usando o modelo de degradação linear por partes"""
    grouped = df.groupby("engine_id")["cycle"].max().reset_index()
    grouped.columns = ["engine_id", "max_cycle"]

    df = df.merge(grouped, on="engine_id", how="left")
    df["RUL"] = df["max_cycle"] - df["cycle"]

    # Aplica o modelo de degradação linear por partes
    df["RUL"] = np.where(df["RUL"] > Re, Re, df["RUL"])

    return df.drop(columns=["max_cycle"])


# Usando Re=128 conforme sugerido no artigo para FD001
Re = 128
df_train_raw = create_rul_labels(df_train_raw, Re)

In [0]:
import matplotlib.pyplot as plt
import seaborn as sns

In [0]:
plot_rul_by_engine(df_train_raw, engines_to_plot=20)

### Preparando as Janelas Temporais

O artigo usa janelas temporais com tamanho (nw) e stride (ns):


separando os dados de validação e treino


In [0]:
from sklearn.model_selection import train_test_split

unique_engines = np.unique(df_train_raw["engine_id"])
train_engines, val_engines = train_test_split(
    unique_engines, test_size=0.2, shuffle=False
)

df_train = df_train_raw[df_train_raw["engine_id"].isin(train_engines)]
df_val = df_train_raw[df_train_raw["engine_id"].isin(val_engines)]
display(train_engines, df_train.shape)
display(val_engines, df_val.shape)

In [0]:
def create_time_windows(df, window_size, window_stride, sensor_cols):
    """Cria janelas temporais dos dados dos sensores e retorna um DataFrame com info da janela"""
    sequences = []
    labels = []
    engine_ids = []
    last_cycles = []

    for engine_id in df["engine_id"].unique():
        engine_data = df[df["engine_id"] == engine_id]
        sensor_data = engine_data[sensor_cols].values
        rul_data = engine_data["RUL"].values

        # Cria janelas deslizantes
        for i in range(0, len(engine_data) - window_size + 1, window_stride):
            window = sensor_data[i : i + window_size]
            label = rul_data[i + window_size - 1]
            sequences.append(window)
            labels.append(label)
            engine_ids.append(engine_id)
            last_cycles.append(engine_data["cycle"].iloc[i + window_size - 1])

    # Flatten sequences for MLP input
    n_samples = len(sequences)
    n_timesteps = window_size
    n_features = len(sensor_cols)
    flattened_sequences = np.array(sequences).reshape(
        (n_samples, n_timesteps * n_features)
    )

    df_windows = pd.DataFrame(
        {
            "engine_id": engine_ids,
            "last_cycle": last_cycles,
            "data_vector": list(flattened_sequences),
            "RUL": labels,
        }
    )

    return df_windows

In [0]:
# Parâmetros do artigo para FD001: nw=24, ns=1
window_size = 30
window_stride = 1

# Criando as sequências de treino
sensor_cols = selected_sensors
df_train_windows = create_time_windows(
    df_train, window_size, window_stride, sensor_cols
)

# Criando as sequências de validação
df_val_windows = create_time_windows(df_val, window_size, window_stride, sensor_cols)

# Separando X_train e y_train
X_train = np.array(list(df_train_windows["data_vector"]))
y_train = df_train_windows["RUL"].values

# Separando X_val e y_val
X_val = np.array(list(df_val_windows["data_vector"]))
y_val = df_val_windows["RUL"].values
val_data = (X_val, y_val)

Salvar os dataframes no catalog

In [0]:
def save_to_catalog(df, table_name):
    df_spark = spark.createDataFrame(df)
    df_spark.write.mode("overwrite").saveAsTable(f"default.{table_name}")